# 07 - Governed RAG And Embedding Ingestion Gate

This notebook checks data before it enters a retrieval or embedding workflow. The agent treats indexing as an AI use, asks Metatate for the governing decision, and only prepares approved context.

The example intentionally includes support-ticket text because it is useful for a chatbot but blocked for model training in the base AcmeCloud policy.


In [ ]:
from pathlib import Path
import json
import os
import sys

import pandas as pd

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

from common import get_client

client = get_client()
mode = os.getenv("METATATE_EXAMPLES_MODE", "offline")
print(f"Metatate examples mode: {mode}")


def decision_label(response):
    data = response.get("data", {})
    decision = data.get("decision")
    if isinstance(decision, dict):
        return decision.get("decision") or decision.get("overall_decision") or data.get("overall_decision", "UNKNOWN")
    return decision or data.get("overall_decision", "UNKNOWN")


def rationale_text(response):
    data = response.get("data", {})
    decision = data.get("decision")
    if isinstance(decision, dict):
        return decision.get("rationale") or data.get("summary") or ""
    return data.get("rationale") or data.get("summary") or ""


def agent_action_text(response):
    action = response.get("data", {}).get("agent_action", {})
    if isinstance(action, dict):
        return action.get("message") or action.get("safe_next_step") or action.get("suggested_next_tool") or ""
    return ""


## Candidate sources for retrieval or embedding


In [ ]:
candidates = [
    {
        "asset": "support_ticket_text",
        "table_name": "ACMECLOUD_DEMO.PUBLIC.SUPPORT_TICKETS",
        "operation": "train",
        "intended_use": "ml_training",
        "actor_role": "ML_ENGINEER",
        "columns": ["TICKET_TEXT", "PRIORITY", "PRODUCT_AREA"],
        "why": "Use support text as examples for a support assistant.",
    },
    {
        "asset": "customer_arr_aggregate",
        "sql": "SELECT region, account_status, SUM(arr) AS arr FROM ACMECLOUD_DEMO.PUBLIC.CUSTOMERS WHERE account_status = 'active' GROUP BY region, account_status",
        "operation": "read",
        "intended_use": "analytics",
        "actor_role": "DATA_ANALYST",
        "why": "Expose aggregate revenue context for retrieval.",
    },
]


## Gate each candidate through Metatate


In [ ]:
def evaluate_candidate(candidate):
    if "sql" in candidate:
        response = client.validate_query_context(
            candidate["sql"],
            operation=candidate["operation"],
            intended_use=candidate["intended_use"],
            actor_role=candidate["actor_role"],
        )
    else:
        response = client.authorize_use(
            candidate["table_name"],
            operation=candidate["operation"],
            intended_use=candidate["intended_use"],
            actor_role=candidate["actor_role"],
            columns=candidate["columns"],
            raw_request_text=candidate["why"],
        )

    decision = decision_label(response)
    if decision == "ALLOW":
        gate = "index"
    elif decision == "CONDITIONAL":
        gate = "prepare_controls_first"
    else:
        gate = "do_not_index"

    return {
        "asset": candidate["asset"],
        "decision": decision,
        "gate": gate,
        "rationale": rationale_text(response),
        "agent_action": agent_action_text(response),
    }

ingestion_plan = pd.DataFrame([evaluate_candidate(candidate) for candidate in candidates])
ingestion_plan


## Build the safe retrieval scope


In [ ]:
safe_scope = ingestion_plan[ingestion_plan["gate"] == "index"]["asset"].tolist()
blocked_scope = ingestion_plan[ingestion_plan["gate"] == "do_not_index"]["asset"].tolist()

print("Safe to index now:")
print(safe_scope)
print("\nBlocked from indexing:")
print(blocked_scope)


The important behavior is pre-ingestion control. Once sensitive text is embedded, deleting or proving non-use is hard. Metatate gives the agent a decision before the index is built.
